In [4]:
import numpy as np
import math
import random


def entropy(rv):
    total = len(rv)
    _, cnts = np.unique(rv, return_counts=True)
    entropy = 0
    for cnt in cnts:
        p_i = cnt / total
        entropy += -(p_i) * math.log2(p_i)
    return entropy


def info_gain(rv, y):
    S = len(y)
    info_gain = entropy(y)
    for val in np.unique(rv):
        mask = rv == val
        info_gain -= np.sum(mask) / S * entropy(y[mask])
    return info_gain


def gain_ratio(rv, y):
    SplitInfo = entropy(rv)
    if SplitInfo == 0:  # indeterminate
        return 0
    return info_gain(rv, y) / SplitInfo


def misclassification(_rv, y):
    max_prob = -math.inf
    S = len(y)
    _, cnts = np.unique(y, return_counts=True)
    for cnt in cnts:
        max_prob = max(max_prob, cnt / S)
    return 1 - max_prob


def gini_val(y):
    S = len(y)
    _, cnts = np.unique(y, return_counts=True)
    gini = 1
    for cnt in cnts:
        gini -= (cnt / S) ** 2
    return gini


def gini_idx(rv, y):
    D = len(rv)
    gini = 0
    for val in np.unique(rv):
        mask = rv == val
        gini += (np.sum(mask) / D) * gini_val(y[mask])
    return gini


class Node:
    def __init__(self, *, terminal=False, label=None, next=None, result=None):
        self.terminal = terminal
        self.label = label
        self.next = {} if next is None else next
        self.result = result


class DecisionTreeClassifier:
    def __init__(self, estimator="info_gain"):
        self.estimator = estimator
        pass

    def estimate(self, rv, y):
        if self.estimator == "misclassification":
            return -misclassification(
                rv, y
            )  # we negate it so that it can fit with our max logic
        elif (
            self.estimator == "gini"
        ):  # we negate it so that it can fit with our max logic better
            return -gini_idx(rv, y)
        elif self.estimator == "gain_ratio":
            return gain_ratio(rv, y)
        else:
            return info_gain(rv, y)

    def construct(self, X, y, possible_cols):
        if len(possible_cols) == 0:
            return Node(terminal=True, result=y[0])  # TODO: improve heuristic?

        S_entropy = entropy(y)

        if S_entropy == 0:
            return Node(terminal=True, result=y[0])  # TODO: improve heuristic?

        best_fit_idx = -1
        best_fit = -math.inf

        for col in possible_cols:
            rv = X[:, col]
            info_gain = self.estimate(rv, y)
            if info_gain > best_fit:
                best_fit_idx = col
                best_fit = info_gain

        winner_rv = X[:, best_fit_idx]
        curr = Node(terminal=False, label=best_fit_idx)
        possible_cols.remove(best_fit_idx)
        for val in np.unique(winner_rv):
            mask = winner_rv == val
            X_new = X[mask]
            y_new = y[mask]
            curr.next[val] = self.construct(X_new, y_new, possible_cols.copy())
        return curr

    def fit(self, X, y):
        # we can pre-process here
        self.root = self.construct(X, y, [i for i in range(X.shape[1])])
        pass

    def classify(self, X):
        y = []
        for row in X:
            curr = self.root
            while not curr.terminal:
                val = row[curr.label]
                if not val in curr.next:
                    # TODO: improve the heuristic
                    val = random.choice(
                        list(iter(curr.next.keys()))
                    )  # INVARIANT: atleast has one value
                curr = curr.next[val]
            y.append(curr.result)
        return y

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier as SklearnDecisionTree
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)


df = pd.read_csv("mushroom.csv")

encoders = {}

for col in df.columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

X = df.drop(columns=["class"]).to_numpy()
y = df["class"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

estimators = [
    ("Information Gain (ID3)", "info_gain"),
    ("Gain Ratio (C4.5)", "gain_ratio"),
    ("Gini Index", "gini"),
    ("Misclassification Error", "misclassification"),
]


def print_metrics(name, y_true, y_pred):
    print(f"\n{name}")
    print("=" * len(name))

    print(f"Accuracy : {accuracy_score(y_true, y_pred)}")
    print(f"Precision: {precision_score(y_true, y_pred)}")
    print(f"Recall   : {recall_score(y_true, y_pred)}")
    print(f"F1 Score : {f1_score(y_true, y_pred)}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))



for name, estimator in estimators:
    tree = DecisionTreeClassifier(estimator=estimator)
    tree.fit(X_train, y_train)
    pred = tree.classify(X_test)

    print_metrics(name, y_test, pred)


sk_tree = SklearnDecisionTree(
    criterion="gini",
    random_state=42,
)

sk_tree.fit(X_train, y_train)
sk_pred = sk_tree.predict(X_test)

print_metrics("Scikit-Learn (Gini)", y_test, sk_pred)


Information Gain (ID3)
Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0

Confusion Matrix:
[[843   0]
 [  0 782]]

Gain Ratio (C4.5)
Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0

Confusion Matrix:
[[843   0]
 [  0 782]]

Gini Index
Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0

Confusion Matrix:
[[843   0]
 [  0 782]]

Misclassification Error
Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0

Confusion Matrix:
[[843   0]
 [  0 782]]

Scikit-Learn (Gini)
Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0

Confusion Matrix:
[[843   0]
 [  0 782]]
